Assignment 11

Fine-Tuning GPT-2 on a Custom Text Style

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import torch

In [3]:
text = """
Hey, are you coming to class today?
No, I have an assignment to finish.
Seriously? It is due today?
Yeah, and I have not even started.
That sounds like you.
I know, do not remind me.
Did you attend the morning class?
Nope, I woke up late.
Again?
Yes, again.
What are you doing now?
I am trying to finish my assignment.
You should have started yesterday.
I know that now.
Are you going to the library?
Maybe after lunch.
I need to study too.
What subject?
Data science.
That subject is difficult.
It is not that bad.
You only say that because you like coding.
Maybe you are right.
Did you finish your project?
Almost.
What is left?
I have to complete the report.
That should not take long.
I hope so.
Are you free this evening?
No, I have an internship call.
At what time?
Around seven.
Then we can talk later.
Sure.
Did you have lunch?
Not yet.
Why?
I was busy with my assignment.
Please eat something first.
Okay, mom.
I am serious.
Fine, I will eat.
What are you doing tomorrow?
I have college.
Of course you do.
And I have a test.
Good luck.
Thanks, I need it.
Did you prepare?
A little.
That means no.
Maybe.
You should study tonight.
I know.
Stop giving me advice.
Then stop asking me questions.
Fair enough.
Are you coming to campus tomorrow?
Yes.
What time?
Around nine.
Meet me near the library.
Okay.
Do not be late.
I will try.
You always say that.
And I am always late.
Exactly.
Did you submit the assignment?
Not yet.
What are you waiting for?
I am checking it one last time.
That is smart.
I hope there are no mistakes.
There probably are.
Do not say that.
I am just being honest.
Sometimes honesty is annoying.
I know.
Are you nervous about the test?
A little.
You will do fine.
I hope so.
Just revise the important topics.
I will.
Do not stay awake all night.
I will try.
You definitely will.
Maybe.
Good night.
Good night.
See you tomorrow.
See you.
Wait.
What?
Do not forget your charger.
Oh right.
Thanks.
You are welcome.
Did you reach college?
Yes, I just arrived.
Where are you?
Near the main block.
I am coming.
Okay.
Did the class start?
Not yet.
Lucky.
I know.
What are we studying today?
Machine learning.
Again?
Yes, again.
At least it is interesting.
True.
Did you understand yesterday's topic?
Mostly.
I did not understand anything.
I can explain it later.
Really?
Sure.
Thanks.
No problem.
Are you hungry?
Very.
Let's get something after class.
Okay.
What do you want?
Maybe a sandwich.
Good choice.
I am getting coffee too.
Of course you are.
Coffee keeps me alive.
That is dramatic.
But true.
How was your test?
Better than expected.
Really?
Yes.
I think I did well.
I told you.
You were right.
I know.
Do not become proud.
Too late.
What are you doing tonight?
Probably studying.
Probably?
Okay, definitely studying.
That sounds better.
I have another assignment.
Again?
College never ends.
True.
We should study together.
That might help.
Let's meet online at eight.
Okay.
Do not forget.
I will not.
You always forget.
Not this time.
We will see.
Good luck with your work.
Thanks.
You too.
I am tired.
Same.
Let's finish everything early.
Good idea.
Then we can relax.
Finally.
Good night.
Good night.
See you tomorrow.
See you.
""".strip()

In [4]:
with open("custom_text.txt", "w") as f:
    f.write(text)

print("Dataset saved successfully!")
print("Number of lines:", len(text.splitlines()))

Dataset saved successfully!
Number of lines: 177


In [5]:
lines = text.splitlines()

dataset = Dataset.from_dict({
    "text": lines
})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 177
})


In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("GPT-2 loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully!


In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Tokenization completed!


In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [9]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.019600
20,3.258106
30,2.787221
40,2.798116
50,3.094201
60,2.869466
70,2.706807
80,2.326085
90,2.244951
100,1.889235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=267, training_loss=2.1179028593199085, metrics={'train_runtime': 822.8736, 'train_samples_per_second': 0.645, 'train_steps_per_second': 0.324, 'total_flos': 17343258624000.0, 'train_loss': 2.1179028593199085, 'epoch': 3.0})

In [10]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Fine-tuned model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved!


In [11]:
base_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
base_model = GPT2LMHeadModel.from_pretrained("gpt2")

base_tokenizer.pad_token = base_tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [12]:
prompt = "Hey, are you coming to class today?"

inputs = base_tokenizer(prompt, return_tensors="pt")

output = base_model.generate(
    **inputs,
    max_length=80,
    do_sample=True,
    temperature=0.8,
    pad_token_id=base_tokenizer.eos_token_id
)

print("ORIGINAL GPT-2 OUTPUT:")
print(base_tokenizer.decode(output[0], skip_special_tokens=True))

ORIGINAL GPT-2 OUTPUT:
Hey, are you coming to class today? Do you want to join us?"

"Yes sir."

"Just as soon as your eyes opened, please go back to school. I'll come and take you with me."

She was about to say, but he did not want to be interrupted.

"Then I'll find other teachers for you, and do as


In [13]:
fine_tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")
fine_model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned")

fine_tokenizer.pad_token = fine_tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
inputs = fine_tokenizer(prompt, return_tensors="pt")

output = fine_model.generate(
    **inputs,
    max_length=80,
    do_sample=True,
    temperature=0.8,
    pad_token_id=fine_tokenizer.eos_token_id
)

print("FINE-TUNED GPT-2 OUTPUT:")
print(fine_tokenizer.decode(output[0], skip_special_tokens=True))

FINE-TUNED GPT-2 OUTPUT:
Hey, are you coming to class today? I should discuss my assignment. Can I finish my assignment later? I hope so. I can relax.

Ruby started studying again. She started studying later. I started studying the morning. Ruby started studying yesterday. I started studying yesterday. I started studying yesterday. I started studying yesterday. I started studying tomorrow. I started studying yesterday. I started


In [15]:
print("BEFORE FINE-TUNING:")
print("Original GPT-2 produces more general and unpredictable text.")

print("\nAFTER FINE-TUNING:")
print("Fine-tuned GPT-2 produces text that is more similar to casual student conversations.")

BEFORE FINE-TUNING:
Original GPT-2 produces more general and unpredictable text.

AFTER FINE-TUNING:
Fine-tuned GPT-2 produces text that is more similar to casual student conversations.


Difference 1 — Writing Style:

The original GPT-2 produces general-purpose text, while the fine-tuned model generates more casual and conversational sentences.


Difference 2 — Topic:

The original model may generate unrelated topics, whereas the fine-tuned model focuses more on student life, classes, assignments, tests, and daily conversations.


Conclusion:

Fine-tuning helped GPT-2 adapt to the custom student-conversation writing style. The fine-tuned model generated responses that were more consistent with the style and topics present in the training dataset.